# 🎯 Recommendation Validation Pipeline (developer / QA)

Prove that **every recommendation is a justified consequence of the reading history** — not a
random article from the catalog. This is the recommendation counterpart to the Metric Validation
notebook: where that one recomputes the dashboard's numbers, this one validates the recommender's
*behaviour and explanations*.

For each scenario (or your own reading history) it checks, PASS / FAIL:

- ✅ **Evidence-backed** — every recommendation resolves to an explanation, and that explanation's
  evidence is a **subset of the reader context** it was handed (the resolver never invents a fact);
- ✅ **Explanations validate** — every explanation's gate re-derives clean, and is exactly one
  sentence (never combined);
- ✅ **Deterministic** — identical history → byte-identical feed + explanations (no hidden randomness);
- ✅ **History-sensitive** — the feed changes when the reading history changes (genuinely personalized).

It ships with **9 scenarios** mapped to explanation types (`same_story`, `same_publisher`,
`follow_up_story`, `new_publisher`, `bridge`, `long_tail`, `mixed_feed`, `cold_start`,
`story_follower`) so a failure names the explanation *type*, not an opaque persona.

**Scope (Phase 1):** this reuses the production graph **and** the production ranking, then validates
behaviour on top. Independently recomputing the RWE-B / RWE-D scores from the graph — the
"prove-the-matrix-multiplication" check — is Phase 2 (21d.2). Nothing here changes the recommender.


In [ ]:
#@title 1 · Setup — clone the repo + install the engine (pure Python; no web / server)
import os, sys, subprocess, pathlib

REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}         # only if the repo is private

def _in_repo():
    return pathlib.Path("examples/rec_pipeline").is_dir()

if not _in_repo():
    auth = (GITHUB_TOKEN + "@") if GITHUB_TOKEN else ""
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://%sgithub.com/%s.git" % (auth, REPO), "app"], check=True)
    os.chdir("app")
print("repo:", pathlib.Path.cwd())

# The pipeline builds REAL recommendation corpora (Backend + Personalizer), so it needs the engine
# library — unlike the Metric Validation notebook, which is pure Python. No web app / server / tunnel.
try:
    import numpy, scipy, sqlalchemy  # noqa: F401
    print("engine deps already present")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[serve]"], check=True)
print("setup done — run cell 2.")


In [ ]:
#@title 2 · Run the validation  (all scenarios by default; or your own reading history)
#@markdown **Golden Personas** validates the 9 built-in scenarios. **My Reading History** validates
#@markdown YOUR exported reads (`/api/me/history` JSON, `{"reads":[...]}`, or `{"scored":...}` rows) —
#@markdown turning the notebook into a debugging tool for a real diet.
MODE = "Golden Personas"  #@param ["Golden Personas", "My Reading History"]
SCENARIO = "all"  #@param ["all", "same_story", "same_publisher", "follow_up_story", "new_publisher", "bridge", "long_tail", "mixed_feed", "cold_start", "story_follower"]
HISTORY_JSON = "/content/my_reads.json"  #@param {type:"string"}
FAST = True  #@param {type:"boolean"}
#@markdown FAST skips the two rebuild-based checks (pipeline determinism + history-sensitivity) for a
#@markdown quick first pass; untick it for the full, slower validation.

import json, subprocess, sys

cmd = [sys.executable, "examples/validate_recs.py", "--report", "json"]
if MODE == "My Reading History":
    cmd += ["--history", HISTORY_JSON]
elif SCENARIO != "all":
    cmd += ["--scenario", SCENARIO]
if FAST:
    cmd += ["--fast"]

proc = subprocess.run(cmd, capture_output=True, text=True)
if not proc.stdout.strip():
    raise RuntimeError("pipeline produced no JSON:\n" + proc.stderr[-2000:])
RUN = json.loads(proc.stdout)

print("=" * 60)
print("MODE   :", MODE, "" if MODE == "My Reading History" else f"· scenario: {SCENARIO}",
      "· fast" if FAST else "· deep")
print("STATUS :", "PASS ✅" if RUN["passed"] else "FAIL ❌", f"  ({RUN['fixtures']} scenario(s))")
print("=" * 60)
for r in RUN["results"]:
    mark = "✅" if r["passed"] else "❌"
    sens = next((c["detail"] for c in r["checks"] if c["check"].startswith("the feed changes")), "")
    tail = f"  · sensitivity: {sens}" if sens else ("  · (cold-start)" if not r["measured"] else "")
    print(f"{mark}  {r['name']:18s}  served={r['served']:2d}{tail}")
    for c in r["checks"]:
        if not c["passed"]:
            print(f"      ↳ FAIL [{c['stage']}] {c['check']} — {c['detail']}")
print("\n(FAST skips the rebuild checks; untick FAST for the full run. Cell 3 shows the full report.)")


In [ ]:
#@title 3 · Full stage-by-stage report (text)
import subprocess, sys

cmd = [sys.executable, "examples/validate_recs.py", "--report", "text"]
if MODE == "My Reading History":
    cmd += ["--history", HISTORY_JSON]
elif SCENARIO != "all":
    cmd += ["--scenario", SCENARIO]
if FAST:
    cmd += ["--fast"]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)


In [ ]:
#@title 4 · Understand one recommendation  (reads → story → recommendation → evidence → validate)
#@markdown Trace the whole chain for one scenario's target article — the exact answer to
#@markdown "why would I get *this* article?", built in-process so you can read every step.
EXPLAIN_SCENARIO = "same_story"  #@param ["same_story", "same_publisher", "follow_up_story", "new_publisher", "bridge", "long_tail", "mixed_feed", "cold_start", "story_follower"]

sys.path.insert(0, "examples")
from rec_pipeline import pipeline, extract
import evidence_resolver as er

case = extract.build(pipeline.load_fixture(EXPLAIN_SCENARIO))
canon = er._canon

print(f"SCENARIO  {case.name}")
print(f"          {case.description}\n")

print("① Reading history (what the reader has read):")
for u in case.reads:
    art = case.catalog_by_url.get(u, {})
    print(f"     · {art.get('publisher','?'):22s} {art.get('topic','?'):12s} {u.split('/')[-1]}")

tr = case.target_rec()
if tr is None:
    print("\n(this scenario has no single target article)")
else:
    art = tr["article"]
    print(f"\n② Candidate recommendation:  {art['publisher']} — {art['headline']!r}")

    story = case.index.get(canon(case.target_url or ""))
    print("\n③ Story cluster:", end=" ")
    if story:
        pubs = sorted({m['publisher'] for m in story['coverage']})
        read_members = [u for u in case.reads if u in {canon(m['url']) for m in story['coverage']}]
        print(f"story {story['storyId'][:12]}… covered by {pubs}")
        print(f"     the reader already read {len(read_members)} article(s) of this story")
    else:
        print("this article is not part of a multi-publisher story")

    exp = er.resolve(tr, case.context, case.index)
    print(f"\n④ Evidence Resolver →  type = {exp['type']}"
          + (f" / {exp['variant']}" if exp.get('variant') else ""))
    print(f"     message: {exp['message']}")
    print(f"     evidence: {json.dumps(exp.get('evidence', {}))[:200]}")

    fails = er.validate(exp, tr, case.context, case.index)
    invented = __import__("rec_pipeline.evidence", fromlist=["evidence_subset_of_context"]) \
        .evidence_subset_of_context(exp, tr, case.context, case.index)
    print(f"\n⑤ validate(): {'PASS ✅' if not fails else 'FAIL ❌ ' + str(fails)}")
    print(f"   evidence ⊆ context: {'PASS ✅ (invents nothing)' if not invented else 'FAIL ❌ ' + str(invented)}")


In [ ]:
#@title 5 · Explanation-type coverage  (which scenario proves which explanation type)
sys.path.insert(0, "examples")
from rec_pipeline import pipeline, extract
import evidence_resolver as er

rows = []
for name in pipeline.fixture_names():
    fx = pipeline.load_fixture(name)
    exp_field = fx.get("expected", {})
    case = extract.build(fx)
    tr = case.target_rec()
    got = er.resolve(tr, case.context, case.index) if tr is not None else {}
    want = exp_field.get("targetType") or (f"NOT {exp_field['targetTypeNot']}"
           if exp_field.get("targetTypeNot") else f">={exp_field.get('minDistinctTypes','?')} types")
    rows.append((name, want, got.get("type", "—"), got.get("variant", "") or ""))

w = max(len(r[0]) for r in rows)
print(f"{'scenario':{w}}  {'expected':16}  {'resolved type':16}  variant")
print("-" * (w + 42))
for name, want, got, var in rows:
    print(f"{name:{w}}  {want:16}  {got:16}  {var}")
print("\nEach row's 'resolved type' is what the Evidence Resolver produced for that scenario's "
      "target — the vocabulary in action. (Run cell 2 for the full PASS/FAIL.)")


## 6 · Scope & what's next

**This notebook (21d Phase 1)** validates recommendation *behaviour and explanations* over the real
production ranking. It deliberately does **not**:

- **Independently recompute the RWE-B / RWE-D scores** from the FeedbackGraph and diff them against
  the production ranking — the "prove the matrix multiplication" engineering check. That is **Phase 2
  (21d.2)**: a fine exercise, but less immediately valuable than proving the recommendations users
  actually see are justified.
- **Story Consistency (Stage 6)** — a dedicated end-to-end check that a `story_match` recommendation
  belongs to the expected story, that the previously-read article is in the same cluster, and that
  the publisher differs — is a natural future addition (the evidence is already there; it would just
  formalize the chain).

**One behaviour this pipeline surfaced:** `topic_continuity` (explanation Priority 2) fires whenever
a recommended article's topic is among the reader's top topics, which pre-empts `new_publisher`,
`bridge`, and `long_tail` for on-topic articles. Whether topic-continuity *should* outrank a
brand-new outlet or a genuine bridge is a product decision — the pipeline has made the behaviour
visible so it can be decided deliberately, not by accident.

**Companion notebooks:** the **Browser Extension Playground** answers *"does the end-to-end product
work?"*; this one answers *"can I prove the recommendations are correct?"*.
